# AI Coach - Interactive Repetition Cropping and Labeling Tool

This notebook provides an interactive tool to precisely crop and label squat repetitions from videos. It's designed to give you full control over defining each rep, resulting in a high-quality, clean dataset.



In [ ]:
%pip -q install ipywidgets
%pip -q install matplotlib

import os
from pathlib import Path
import sys
import cv2
import pandas as pd
import numpy as np
import ipywidgets as widgets
from ipywidgets import HBox, VBox, Layout
from IPython.display import display, clear_output
from tqdm.auto import tqdm
import base64

PROJECT_ROOT = Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

from ai_coach.pose import PoseEstimator, compute_basic_angles
from ai_coach.per_rep_features import RepAggregate, SQUAT_REP_FEATURES



Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


In [ ]:
KAGGLE_PATH = Path(f'{os.path.expanduser("~")}/.cache/kagglehub/datasets/hasyimabdillah/workoutfitness-video/versions/5')
YOUTUBE_PATH = Path('data/youtube_videos')
VIDEO_EXTS = {'.mp4', '.avi', '.mov', '.mkv'}
OUTPUT_CSV = Path('outputs/manual_labels.csv')
NEEDS_CROPPING_LOG = Path('outputs/needs_cropping.txt')
OUTPUT_CSV.parent.mkdir(exist_ok=True, parents=True)

EXCLUDE_VIDEOS = {
    "How to Deadlift： 5 Simple Steps.mp4"
}


In [ ]:
class LabelValidator:
    def __init__(self, labels_df, video_base_paths):
        self.df = labels_df.copy()
        self.video_base_paths = video_base_paths
        self.current_index = 0
        
        self.frames = []  
        self.video_frames_cache = []  
        self.current_video_name_cache = None  

        self.play = widgets.Play(value=0, min=0, max=100, step=1, interval=100, description="Press play", disabled=True)
        self.slider = widgets.IntSlider(value=0, min=0, max=100, step=1, description='Frame:', continuous_update=False)
        self.image_widget = widgets.Image(format='jpeg')
        self.info_widget = widgets.HTML(value="Info")
        self.output_widget = widgets.Output()
        
        self.relabel_buttons = VBox([
            widgets.Button(description="Correct", button_style='success', icon='check'),
            widgets.Button(description="Change to Good", button_style='primary'),
            widgets.Button(description="Change to Too High", button_style='warning'),
            widgets.Button(description="Change to Too Low", button_style='warning'),
            widgets.Button(description="Change to Bad Posture", button_style='danger')
        ])
        
        self.control_buttons = HBox([
            widgets.Button(description="<< Prev Rep", button_style='info'),
            widgets.Button(description="Next Rep >>", button_style='info'),
            widgets.Button(description="Save & Close", button_style='primary', icon='save')
        ])

        widgets.jslink((self.play, 'value'), (self.slider, 'value'))
        self.slider.observe(lambda change: self.update_image(change['new']), names='value')
        
        self.relabel_buttons.children[0].on_click(lambda b: self.on_relabel_click(None))
        self.relabel_buttons.children[1].on_click(lambda b: self.on_relabel_click('good'))
        self.relabel_buttons.children[2].on_click(lambda b: self.on_relabel_click('too_high'))
        self.relabel_buttons.children[3].on_click(lambda b: self.on_relabel_click('too_low'))
        self.relabel_buttons.children[4].on_click(lambda b: self.on_relabel_click('bad_posture'))
        
        self.control_buttons.children[0].on_click(self.on_prev_rep_click)
        self.control_buttons.children[1].on_click(self.on_next_rep_click)
        self.control_buttons.children[2].on_click(self.on_close_click)

    def _set_all_controls_disabled(self, disabled):
        """Disables or enables all interactive UI components."""
        self.play.disabled = disabled
        self.slider.disabled = disabled
        for btn in self.relabel_buttons.children:
            btn.disabled = disabled
        self.control_buttons.children[0].disabled = disabled 
        self.control_buttons.children[1].disabled = disabled 

    def _get_video_path(self, video_name):
        for base_path in self.video_base_paths:
            possible_paths = list(base_path.rglob(f"**/{video_name}"))
            if possible_paths:
                return possible_paths[0]
        return None

    def load_rep_segment(self):
        self._set_all_controls_disabled(True)
        
        if self.current_index >= len(self.df):
            self.info_widget.value = "All reps validated!"
            self.on_close_click(None)
            return

        row = self.df.iloc[self.current_index]
        video_name = row['video_source']
        start_frame, end_frame = int(row['start_frame']), int(row['end_frame'])
        
        self.info_widget.value = (f"<b>Rep {self.current_index + 1}/{len(self.df)}</b><br>"
                                  f"Video: {video_name}<br>"
                                  f"Frames: {start_frame}-{end_frame}<br>"
                                  f"Current Label: <b>{row['label']}</b>")

        if video_name != self.current_video_name_cache:
            self.video_frames_cache = [] 
            
            video_path = self._get_video_path(video_name)
            if not video_path:
                with self.output_widget:
                    clear_output(wait=True)
                    print(f"ERROR: Video '{video_name}' not found. Skipping rep.")
                self.on_next_rep_click(None)
                return

            cap = cv2.VideoCapture(str(video_path))
            if not cap.isOpened():
                with self.output_widget:
                    clear_output(wait=True)
                    print(f"ERROR: Could not open video file '{video_path}'. Skipping rep.")
                self.on_next_rep_click(None)
                return
            
            loaded_frames = []
            total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
            pbar = tqdm(total=total_frames, desc=f"Loading {video_name}")
            while cap.isOpened():
                ret, frame = cap.read()
                if not ret: break
                loaded_frames.append(frame)
                pbar.update(1)
            pbar.close()
            cap.release()
            
            self.video_frames_cache = loaded_frames
            self.current_video_name_cache = video_name

        self.frames = self.video_frames_cache[start_frame : end_frame + 1]

        if not self.frames:
            with self.output_widget:
                print(f"Warning: No frames found for rep {self.current_index+1} in '{video_name}' range {start_frame}-{end_frame}. Skipping.")
            import time
            time.sleep(2)
            self.on_next_rep_click(None)
            return

        self.play.max = len(self.frames) - 1
        self.slider.max = len(self.frames) - 1
        self.slider.value = 0
        self.update_image(0)
        
        self._set_all_controls_disabled(False)
    
    def update_image(self, frame_index):
        if 0 <= frame_index < len(self.frames):
            frame = self.frames[frame_index]
            _, encoded_image = cv2.imencode('.jpeg', frame)
            self.image_widget.value = encoded_image.tobytes()

    def on_relabel_click(self, new_label):
        row_index = self.df.index[self.current_index] 
        original_label = self.df.at[row_index, 'label']
        
        if new_label is None: 
            with self.output_widget:
                clear_output(wait=True)
                print(f"Confirmed rep {self.current_index + 1} as '{original_label}'.")
        else:
            self.df.at[row_index, 'label'] = new_label
            with self.output_widget:
                clear_output(wait=True)
                print(f"Changed rep {self.current_index + 1} from '{original_label}' to '{new_label}'.")
        
        self.on_next_rep_click(None)

    def on_next_rep_click(self, button):
        self.current_index += 1
        self.load_rep_segment()

    def on_prev_rep_click(self, button):
        self.current_index = max(0, self.current_index - 1)
        self.load_rep_segment()

    def on_close_click(self, button=None):
        self.df.to_csv(OUTPUT_CSV, index=False)
        for w in [self.play, self.slider, *self.relabel_buttons.children, *self.control_buttons.children]:
            w.disabled = True
        with self.output_widget:
            clear_output(wait=True)
            print(f"Validation finished. Saved updated labels to {OUTPUT_CSV}.")

    def display_widgets(self):
        player = HBox([self.play, self.slider])
        layout = VBox([
            self.info_widget,
            player,
            self.image_widget,
            self.control_buttons,
            self.relabel_buttons,
            self.output_widget
        ])
        display(layout)
        self.load_rep_segment()

def run_labeler():
    if 'all_videos' in globals() and all_videos:
        labeler = VideoLabeler(all_videos)
        labeler.display_widgets()
    else:
        print("No videos found for labeling. Please check your data paths.")

def run_validator():
    if not OUTPUT_CSV.exists() or pd.read_csv(OUTPUT_CSV).empty:
        print(f"Cannot run validator: '{OUTPUT_CSV}' is missing or empty.")
        return
        
    labels_df = pd.read_csv(OUTPUT_CSV)
    
    videos_to_exclude = ['squat_1.MOV', 'squat_2.MOV']
    original_count = len(labels_df)
    labels_df = labels_df[~labels_df['video_source'].isin(videos_to_exclude)].reset_index(drop=True)
    
    if len(labels_df) < original_count:
        print(f"Excluding videos {videos_to_exclude} from validation. {original_count - len(labels_df)} reps will be skipped.")

    if labels_df.empty:
        print("No reps left to validate after exclusions.")
        return

    video_base_paths = [KAGGLE_PATH, YOUTUBE_PATH]
    validator = LabelValidator(labels_df, video_base_paths)
    validator.display_widgets()

tool_chooser = widgets.ToggleButtons(
    options=['Create New Labels', 'Validate Existing Labels'],
    description='Select Tool:',
    disabled=False,
    button_style='',
)

run_button = widgets.Button(description="Run Selected Tool", button_style="success")
chooser_output = widgets.Output()

def on_run_button_clicked(b):
    with chooser_output:
        clear_output(wait=True)
        choice = tool_chooser.value
        if choice == 'Create New Labels':
            print("Starting the Video Labeler...")
            run_labeler()
        elif choice == 'Validate Existing Labels':
            print("Starting the Label Validator...")
            run_validator()

run_button.on_click(on_run_button_clicked)

display(VBox([tool_chooser, run_button, chooser_output]))



In [ ]:

if NEEDS_CROPPING_LOG.exists():
    with open(NEEDS_CROPPING_LOG, 'r') as f:
        skipped_for_cropping = {line.strip() for line in f}
    EXCLUDE_VIDEOS.update(skipped_for_cropping)

kaggle_videos = [p for p in KAGGLE_PATH.rglob('*') if p.suffix.lower() in VIDEO_EXTS and 'squat' in p.as_posix().lower() and p.name not in EXCLUDE_VIDEOS]
youtube_videos = [p for p in YOUTUBE_PATH.rglob('*') if p.suffix.lower() in VIDEO_EXTS and p.name not in EXCLUDE_VIDEOS]
all_videos = kaggle_videos + youtube_videos

print(f"Found {len(all_videos)} videos to label after excluding {len(EXCLUDE_VIDEOS)} files.")


class VideoLabeler:
    def __init__(self, video_paths):
        self.video_paths = video_paths
        self.video_index = 0
        self.cap = None
        self.frames = []
        self.labeled_rows = []
        self.labeled_df = pd.DataFrame()
        
        if OUTPUT_CSV.exists():
            self.labeled_df = pd.read_csv(OUTPUT_CSV)
            if not self.labeled_df.empty:
                self.labeled_rows = self.labeled_df.to_dict('records')
                last_video_labeled = self.labeled_df['video_source'].iloc[-1]
                
                video_names = [p.name for p in self.video_paths]
                try:
                    last_index = video_names.index(last_video_labeled)
                    self.video_index = last_index
                    print(f"Resuming from last labeled video: {last_video_labeled} (video {self.video_index + 1}/{len(self.video_paths)})")
                except ValueError:
                    print(f"Warning: Last labeled video '{last_video_labeled}' not found. Starting from the beginning.")
            
            print(f"Loaded {len(self.labeled_rows)} existing labels from {OUTPUT_CSV}")
        
        self.pose_estimator = PoseEstimator()

        self.play = widgets.Play(value=0, min=0, max=100, step=1, interval=100, description="Press play", disabled=True)
        self.slider = widgets.IntSlider(value=0, min=0, max=100, step=1, description='Frame:', continuous_update=False)
        self.range_slider = widgets.IntRangeSlider(value=[0, 10], min=0, max=100, step=1, description='Crop Rep:', continuous_update=False, layout=Layout(width='80%'))
        self.image_widget = widgets.Image(format='jpeg')
        
        self.label_buttons = VBox([
            widgets.Button(description="Good", button_style='success'),
            widgets.Button(description="Too High", button_style='warning'),
            widgets.Button(description="Too Low", button_style='warning'),
            widgets.Button(description="Bad Posture", button_style='danger')
        ])
        
        self.close_button = widgets.Button(description="Save & Close", button_style='primary', icon='save')
        self.skip_crop_button = widgets.Button(description="Skip (Needs Crop)", button_style='info')
        self.control_buttons = HBox([
            widgets.Button(description="Next Video", button_style='primary'),
            widgets.Button(description="<<", layout=Layout(width='50px')),
            widgets.Button(description=">>", layout=Layout(width='50px')),
            self.skip_crop_button,
            self.close_button
        ])
        
        self.video_name_widget = widgets.HTML(value="No video loaded.")
        self.output_widget = widgets.Output()

        widgets.jslink((self.play, 'value'), (self.slider, 'value'))
        self.slider.observe(self.on_slider_change, names='value')
        
        for button in self.label_buttons.children:
            button.on_click(self.on_label_button_click)
            
        self.control_buttons.children[0].on_click(self.on_next_video_click)
        self.control_buttons.children[1].on_click(self.on_prev_frame_click)
        self.control_buttons.children[2].on_click(self.on_next_frame_click)
        self.skip_crop_button.on_click(self.on_skip_crop_click)
        self.close_button.on_click(self.on_close_click)

        self.load_video()

    def _set_all_controls_disabled(self, disabled):
        """Disables or enables all interactive UI components."""
        self.play.disabled = disabled
        self.slider.disabled = disabled
        self.range_slider.disabled = disabled
        for btn in self.label_buttons.children:
            btn.disabled = disabled
        for btn in self.control_buttons.children:
            btn.disabled = disabled

    def _load_frames(self):
        """Loads all frames from the current video file into memory."""
        if self.cap:
            self.cap.release()
        
        path = self.video_paths[self.video_index]
        self.frames = []
        
        try:
            self.cap = cv2.VideoCapture(str(path))
            if not self.cap.isOpened():
                raise IOError(f"Cannot open video file: {path.name}")

            total_frames = int(self.cap.get(cv2.CAP_PROP_FRAME_COUNT))
            if total_frames <= 0:
                raise ValueError(f"Video file has no frames or is corrupted: {path.name}")
            
            with tqdm(total=total_frames, desc=f"Loading {path.name}") as pbar:
                while self.cap.isOpened():
                    ret, frame = self.cap.read()
                    if not ret:
                        break
                    self.frames.append(frame)
                    pbar.update(1)
            return True
        except (cv2.error, IOError, ValueError) as e:
            if hasattr(self, 'output_widget') and self.output_widget:
                with self.output_widget:
                    clear_output(wait=True)
                    print(f"ERROR: Could not load video '{path.name}'. It may be corrupted. Skipping.")
                    print(f"  Details: {e}")
            else: 
                 print(f"ERROR: Could not load video '{path.name}'. It may be corrupted. Skipping.")
            self.frames = []
            return False
        finally:
            if self.cap:
                self.cap.release()

    def load_video(self):
        """Loads a video and updates the UI, skipping corrupted files."""
        self._set_all_controls_disabled(True) 
        
        
        if self.video_index >= len(self.video_paths):
            self.video_name_widget.value = "All videos processed!"
            self.on_close_click(None) 
            return

        while self.video_index < len(self.video_paths):
            success = self._load_frames()
            if success:
                break 
            else:
                self.video_index += 1 

        if not self.frames:
            self.video_name_widget.value = "Could not load any more videos. All remaining files may be corrupted."
            self.on_close_click(None)
            return
            
        path = self.video_paths[self.video_index]
        self.video_name_widget.value = f"<b>({self.video_index + 1}/{len(self.video_paths)})</b> {path.name}"
        
        num_frames = len(self.frames)
        self.play.max = num_frames - 1
        self.slider.max = num_frames - 1
        self.range_slider.max = num_frames - 1
        self.range_slider.value = [0, min(10, num_frames - 1)]
        
        self.slider.value = 0
        self.update_image(0)
        
        self._set_all_controls_disabled(False) 
        self.close_button.disabled = False 
        self.skip_crop_button.disabled = False

    def on_slider_change(self, change):
        self.update_image(change['new'])

    def update_image(self, frame_index):
        if 0 <= frame_index < len(self.frames):
            frame = self.frames[frame_index]
            _, encoded_image = cv2.imencode('.jpeg', frame)
            self.image_widget.value = encoded_image.tobytes()

    def on_label_button_click(self, button):
        label = button.description.lower().replace(" ", "_")
        start_frame, end_frame = self.range_slider.value
        
        rep_agg = RepAggregate()
        
        for i in range(start_frame, end_frame + 1):
            frame = self.frames[i]
            results, _ = self.pose_estimator.process(frame)
            if results and results.pose_landmarks:
                rep_agg.update(compute_basic_angles(results.pose_landmarks.landmark), results.pose_landmarks.landmark)

        if not rep_agg.frame_count > 0:
            with self.output_widget:
                clear_output(wait=True)
                print(f"Label '{label}' NOT saved. No poses detected in the selected range [{start_frame}, {end_frame}].")
            return

        vec = rep_agg.to_feature_vector()
        row_dict = {SQUAT_REP_FEATURES[i]: float(vec[i]) for i in range(len(SQUAT_REP_FEATURES))}
        row_dict['label'] = label
        row_dict['video_source'] = self.video_paths[self.video_index].name
        row_dict['start_frame'] = start_frame
        row_dict['end_frame'] = end_frame
        
        self.labeled_rows.append(row_dict)
        pd.DataFrame(self.labeled_rows).to_csv(OUTPUT_CSV, index=False)
        
        with self.output_widget:
            clear_output(wait=True)
            print(f"Labeled rep {len(self.labeled_rows)} as '{label}' from range [{start_frame}, {end_frame}]. Saved to CSV.")
    
    def on_skip_crop_click(self, button):
        """Logs the current video for later cropping and moves to the next one."""
        video_path = self.video_paths[self.video_index]
        with open(NEEDS_CROPPING_LOG, 'a') as f:
            f.write(f"{video_path.name}\n")
        
        with self.output_widget:
            clear_output(wait=True)
            print(f"Logged '{video_path.name}' to {NEEDS_CROPPING_LOG}. It will be skipped in future sessions.")
        
        self.on_next_video_click(None)

    def on_close_click(self, button=None):
        """Safely closes the labeler, saves data, and releases resources."""
        if self.labeled_rows:
            pd.DataFrame(self.labeled_rows).to_csv(OUTPUT_CSV, index=False)
        
        self.pose_estimator.close()
        
        self.play.disabled = True
        self.slider.disabled = True
        self.range_slider.disabled = True
        for btn in self.label_buttons.children:
            btn.disabled = True
        for btn in self.control_buttons.children:
            btn.disabled = True
            
        with self.output_widget:
            clear_output(wait=True)
            print(f"Labeling session closed. {len(self.labeled_rows)} labels saved to {OUTPUT_CSV}.")
            print("Pose estimator resources have been released. You can now safely shut down the kernel.")

    def on_next_video_click(self, button):
        self.video_index += 1
        self.load_video()
        
    def on_prev_frame_click(self, button):
        self.slider.value = max(0, self.slider.value - 1)
        
    def on_next_frame_click(self, button):
        self.slider.value = min(self.slider.max, self.slider.value + 1)

    def display_widgets(self):
        player = HBox([self.play, self.slider])
        top_controls = VBox([self.video_name_widget, player, self.range_slider, self.control_buttons])
        main_layout = HBox([VBox([self.image_widget]), VBox([self.label_buttons, self.output_widget])], layout=Layout(align_items='flex-start'))
        
        display(VBox([top_controls, main_layout]))

if all_videos:
    labeler = VideoLabeler(all_videos)
    labeler.display_widgets()
else:
    print("No videos found. Please check your data paths.")

